In [3]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../') # 允许导入 src 目录下的模块

import pandas as pd
import numpy as np
from src.processing import preprocess_data
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_val_score

# 1. 加载原始数据
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

# 2. 一键特征工程
train_final, test_final = preprocess_data(train, test)

# 3. 准备模型输入
X = train_final.drop(['SalePrice', 'Id'], axis=1)
y = train_final['SalePrice']

nan_cols = X.columns[X.isnull().any()].tolist()
print(f"含有缺失值的列: {nan_cols}")

# 如果有，看看到底有多少个缺失值
if nan_cols:
    print(X[nan_cols].isnull().sum())

# 4. 简单的交叉验证看看效果 (以 Ridge 回归为例)
model = Ridge(alpha=10)
scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')
rmse_scores = np.sqrt(-scores)

print(f"RMSE 平均分: {rmse_scores.mean():.4f}")

含有缺失值的列: []
RMSE 平均分: 0.1397


In [4]:
model_ridge = Ridge(alpha=10)
model_ridge.fit(X, y)

# 2. 对测试集进行预测 (记得 X_test 要去掉 Id)
X_test = test_final.drop(['Id'], axis=1)
# 别忘了要把 log 过的价格还原回去！使用 expm1 (exp(x)-1)
predictions = np.expm1(model_ridge.predict(X_test))

# 3. 构造提交文件
submission = pd.DataFrame({
    "Id": test_final["Id"],
    "SalePrice": predictions
})

# 4. 保存到 submission 文件夹
submission.to_csv('../submissions/ridge_submission.csv', index=False)

In [5]:
model_lasso = Lasso(alpha=0.0005, random_state=1)
model_lasso.fit(X, y)

X_test = test_final.drop(['Id'], axis=1)
predictions = np.expm1(model_lasso.predict(X_test))

submission = pd.DataFrame({
    "Id": test_final["Id"],
    "SalePrice": predictions
})

submission.to_csv('../submissions/lasso_submission_v1.csv', index=False)

D:\anaconda\envs\pytorch-env\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.871e-01, tolerance: 2.328e-02
  model = cd_fast.enet_coordinate_descent(


In [8]:
import xgboost as xgb

model_xgb = xgb.XGBRegressor(
    learning_rate=0.05,      # 步长中等，兼顾速度和精度
    n_estimators=3000,       # 足够多的树，配合早停使用
    max_depth=4,             # 限制深度，防止在小样本上过拟合
    min_child_weight=1.5,    # 每个叶子节点所需的最小权重和
    gamma=0,                 # 基础分裂阈值
    subsample=0.7,           # 每次只用 70% 的数据，增加鲁棒性
    colsample_bytree=0.7,    # 每次只用 70% 的特征
    reg_alpha=0.00006,       # 轻微的 L1 正则
    n_jobs=-1,               # 调用所有 CPU 核心
    random_state=42
)

scores = cross_val_score(model_xgb, X, y, cv=5, scoring='neg_mean_squared_error')
print(f"XGBoost RMSE: {np.sqrt(-scores).mean():.4f}")

XGBoost RMSE: 0.1218
